In [42]:
import os  # Used for file management (removing files, path operations)
import pandas as pd  # Used for handling tabular data (reading/writing CSV files)
import subprocess  # Used to run hmm software as an external process
from tqdm import tqdm  # Used to display a progress bar for tracking processing
from os import listdir  # Used to list files in a directory
from Bio import SeqIO  # Used to read and parse FASTA files
from Bio.Seq import Seq  # Used for reverse complementing sequences
from Bio.SearchIO import HmmerIO

import json

import sys
sys.path.append('/home/dylan33smith/projects/Yuzhen/PB_interactions/code')

%load_ext autoreload
%autoreload 2
from rbp_detection import single_hmm_scan, process_protein_fastas, print_hmm_results

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


**HMM database**: The pfam_path points to a .hmm file containing hmm profiles for many domains, including those in the domains list. This file is 'pressed' to optimize it for storage.

**Scanning**: For each gene sequence, run HMMERs hmmscan which;
    - Aligns the sequence against all HMM profiles in the database
    - returns hits (matching domains), scores, biases, and ranges

**Scoring**: Filter the results to only record scores for domains in the domains list.
    - If a domain from domains list is detected, its score is stored, otherwise, it remains 0.
    - The result is a feature matrix where rows are genes and columns are domain scores.

**Feature Integration**: These scores are concatenated with protein embeddings and fed into the XGBoost model to predict RBPs. The HMM score indicate the presence and strenth of RBP-related domains, enhancing the model's ability to classify proteins.

In [3]:
dir_path = '/home/dylan33smith/projects/Yuzhen/PB_interactions/'
phage_file = 'data/phage_genomes/A1a.fasta'
hmm_path = '/home/dylan33smith/src/hmmer-3.4/src'
pfam_path = '/home/dylan33smith/projects/Yuzhen/PB_interactions/hmm_files/RBPdetect_phageRBPs.hmm'

xgb_path = 'RBPdetect_xgb_hmm.json'
phage_path = os.path.abspath(os.path.join(dir_path, phage_file))

genebase = pd.read_csv('data/annotated/genebase_embeddings.csv')

In [29]:
genebase.head()

,phage_ID,gene_ID,gene_sequence,protein_sequence,protein_embedding
0,A1a,A1a_gp1,TTAGACGCTGTGAACCTGACGTTAGAAGCCCTGGGGGAGTCTCGCG...,LDAVNLTLEALGESRVMDINTSNPSAGLARSALARNRRGLLSTGYW...,"[0.04319863021373749, 0.0043669892475008965, 0..."
1,A1a,A1a_gp2,ATGGCGCAATCATTAGAAGGCACCATTCAGAGTCTGCTCCAGGGCG...,MAQSLEGTIQSLLQGVSQQIPRERQPGQLGAQLNMLSDPVSGLRRR...,"[0.0487942099571228, -0.032444849610328674, 0...."
2,A1a,A1a_gp3,ATGGCTATGTGGTGGGCTGTCGCCGCCCTGGCAGGCTCTAAGCTGC...,MAMWWAVAALAGSKLLGAGAQIEVSKARNKAVIQQTAKQLNDIALQ...,"[0.029753318056464195, -0.047349002212285995, ..."
3,A1a,A1a_gp4,ATGCCTGTAATTCAACCCAACCGACAGGGTCTAAATATCGGCGGCG...,MPVIQPNRQGLNIGGVQLQANEVNLPSTVGDVAVDTSKANRLAALA...,"[0.010439022444188595, -0.06183784827589989, -..."
4,A1a,A1a_gp5,ATGGCTCAGTTTCTGAACCAAGAACCGAATCCACAGGAAAAGGATT...,MAQFLNQEPNPQEKDSAKGATLKPAPESVDWNDAGDAGLNALQRSS...,"[0.00653857784345746, -0.06143265217542648, -0..."


In [20]:
# to load json pfam domains
with open('rbp_domains.json', 'r') as f:
    data = json.load(f)

domains = data['new_blocks']
print(domains)


['Phage_T7_tail', 'Tail_spike_N', 'Prophage_tail', 'BppU_N', 'Mtd_N', 'Head_binding', 'DUF3751', 'End_N_terminal', 'phage_tail_N', 'Prophage_tailD1', 'DUF2163', 'Phage_fiber_2', 'unknown_N0', 'unknown_N1', 'unknown_N2', 'unknown_N3', 'unknown_N4', 'unknown_N6', 'unknown_N10', 'unknown_N11', 'unknown_N12', 'unknown_N13', 'unknown_N17', 'unknown_N19', 'unknown_N23', 'unknown_N24', 'unknown_N26', 'unknown_N29', 'unknown_N36', 'unknown_N45', 'unknown_N48', 'unknown_N49', 'unknown_N53', 'unknown_N57', 'unknown_N60', 'unknown_N61', 'unknown_N65', 'unknown_N73', 'unknown_N82', 'unknown_N83', 'unknown_N101', 'unknown_N114', 'unknown_N119', 'unknown_N122', 'unknown_N163', 'unknown_N174', 'unknown_N192', 'unknown_N200', 'unknown_N206', 'unknown_N208', 'Lipase_GDSL_2', 'Pectate_lyase_3', 'gp37_C', 'Beta_helix', 'Gp58', 'End_beta_propel', 'End_tail_spike', 'End_beta_barrel', 'PhageP22-tail', 'Phage_spike_2', 'gp12-short_mid', 'Collar', 'unknown_C2', 'unknown_C3', 'unknown_C8', 'unknown_C15', 'unkn

In [17]:
def hmmpress(hmm_path, pfam_file):
    """
    Prepares an HMM profiles database (pfam_file) for efficient querying by HMMERs hmmscan.
    - This is a one-time setup.
    - hmmpress compressses and indexes the .hmm file, generating auxilary files (.h3m, .h3i) needed for fast searches.

    Inputs:
        hmm_path: path to HMMER software
        pfam_file: Path to the HMM profiles 
    """
    cd_str = "cd " + hmm_path 
    press_str = 'hmmpress ' + pfam_file
    command = cd_str + ';' + press_str
    press_process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    stdout, _ = press_process.communicate()
    if press_process.returncode != 0:
        raise RuntimeError(f"Error running hmmpress: {stdout.decode()} ")

In [ ]:
######### ONLY HAVE TO RUN THIS ONE TIME TO GET HMM PRESSED FILES ################################
# hmm_path = '/home/dylan33smith/src/hmmer-3.4/src'
# pfam_path = '/home/dylan33smith/projects/Yuzhen/PB_interactions/hmm_files/RBPdetect_phageRBPs.hmm'

# hmmpress(hmm_path, pfam_path)

In [46]:
dir_path = '/home/dylan33smith/projects/Yuzhen/PB_interactions'
hmm_path = '/home/dylan33smith/src/hmmer-3.4/src'
pfam_path = '/home/dylan33smith/projects/Yuzhen/PB_interactions/hmm_files/RBPdetect_phageRBPs.hmm'


results = process_protein_fastas(dir_path, hmm_path, pfam_path)
print_hmm_results(results)

Processing protein files: 100%|██████████| 105/105 [00:07<00:00, 13.64it/s]


Processing K74PH129C2_proteins.fasta...

Query: K74PH129C2_gp1, Hits: 0

Query: K74PH129C2_gp2, Hits: 0

Query: K74PH129C2_gp3, Hits: 0

Query: K74PH129C2_gp4, Hits: 0

Query: K74PH129C2_gp5, Hits: 0

Query: K74PH129C2_gp6, Hits: 0

Query: K74PH129C2_gp7, Hits: 0

Query: K74PH129C2_gp8, Hits: 0

Query: K74PH129C2_gp9, Hits: 0

Query: K74PH129C2_gp10, Hits: 1
  Hit ID: Phage_T7_tail, E-value: 1.9e-36

Query: K74PH129C2_gp11, Hits: 1
  Hit ID: Pectate_lyase_3, E-value: 5.7e-06

Query: K74PH129C2_gp12, Hits: 0

Query: K74PH129C2_gp13, Hits: 0

Query: K74PH129C2_gp14, Hits: 0

Query: K74PH129C2_gp15, Hits: 0

Query: K74PH129C2_gp16, Hits: 0

Query: K74PH129C2_gp17, Hits: 0

Query: K74PH129C2_gp18, Hits: 0

Query: K74PH129C2_gp19, Hits: 0

Query: K74PH129C2_gp20, Hits: 0

Query: K74PH129C2_gp21, Hits: 0

Query: K74PH129C2_gp22, Hits: 0

Query: K74PH129C2_gp23, Hits: 0

Query: K74PH129C2_gp24, Hits: 0

Query: K74PH129C2_gp25, Hits: 0

Query: K74PH129C2_gp26, Hits: 0

Query: K74PH129C2_gp27,

In [47]:
def save_results(results_dict, output_file):
    """
    Save HMM results to a JSON file.
    """
    with open(output_file, 'w') as f:
        json.dump(results_dict, f, indent=2)

def load_results(input_file):
    """
    Load HMM results from a JSON file.
    """
    with open(input_file) as f:
        return json.load(f)

In [48]:
save_results(results, 'hmm_results.json')